# Face Anonymization from 5 Given Landmarks

Pure-geometric face removal for Einstar photogrammetry scans. Given the 5
anatomical landmarks (Nz, Iz, Cz, Lpa, Rpa), `anonymize_scan` strips the face
and reverts the result back to the raw Einstar (`crs=digitized`) frame so the
saved `.obj` is a drop-in replacement for the original at co-registration time.

Flow: load &rarr; pick 5 landmarks &rarr; `anonymize_scan` &rarr; save.


In [ ]:
import logging
import os
from pathlib import Path

import numpy as np
import pyvista as pv
import xarray as xr
from PIL import Image
import trimesh

import cedalion
import cedalion.dataclasses as cdc
import cedalion.io
import cedalion.vis.blocks
from cedalion.vtktutils import trimesh_to_vtk_polydata
from cedalion.geometry.photogrammetry.anonymization import (
    anonymize_scan,
    save_anonymized_scan,
)

logging.getLogger('cedalion').setLevel(logging.WARNING)
pv.set_jupyter_backend('server')

# === Options ===
# Path to your Einstar .obj scan. The notebook expects a sibling .jpg
# texture (Einstar exports both).
SCAN_PATH = './scan.obj'         # edit me
OUT_PATH = None                  # None -> write '<stem>_anon.obj' next to SCAN_PATH

# True: run the picker. False: load landmarks from a TSV sidecar (set below).
INTERACTIVE = True
CACHED_LANDMARKS_TSV = None      # used only when INTERACTIVE = False


# === Mask parameters (edit to adjust the anonymized region) ===
EAR_DELETE_RADIUS_MM = 40.0      # sphere radius around LPA/RPA included in deletion
LANDMARK_KEEP_RADIUS_MM = 8.0    # sphere radius around each landmark kept intact
EYEBROW_OFFSET_MM = 10.0         # failsafe cap height above Nz (flush-cap fallback)
CAP_Z_CEILING_MM = 40.0          # mm above Nz where cap detection is clamped
HEAD_ISOLATION_RADIUS_MM = 220.0 # sphere radius used to strip body/shoulders


## Load the Einstar scan


In [9]:
surface = cedalion.io.read_einstar_obj(SCAN_PATH)
print(f'Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces')

if OUT_PATH is None:
    p = Path(SCAN_PATH)
    OUT_PATH = str(p.with_name(p.stem + '_anon' + p.suffix))

# trimesh 4.6 places the texture image on visual.material.image; if neither
# path has it, attach it from the sibling JPG so save_anonymized_scan can
# sanitize the final texture.
visual = surface.mesh.visual
img = getattr(visual, 'image', None) or getattr(getattr(visual, 'material', None), 'image', None)
if img is None:
    jpg = SCAN_PATH.replace('.obj', '.jpg')
    uv = getattr(visual, 'uv', None)
    assert os.path.exists(jpg) and uv is not None, 'no texture available'
    surface.mesh.visual = trimesh.visual.TextureVisuals(
        uv=uv, image=Image.open(jpg).convert('RGBA'),
    )


Loaded: 943,539 vertices, 1,736,738 faces


## Pick the 5 landmarks

Right-click on the mesh to place a sphere; click a sphere to cycle its label
through `Nz -> Iz -> Cz -> Lpa -> Rpa`. Close the window when all 5 are placed.
Skip this cell if `INTERACTIVE = False`.


In [10]:
if INTERACTIVE:
    pvplt = pv.Plotter()
    get_landmarks = cedalion.vis.blocks.plot_surface(
        pvplt, surface, opacity=1.0, pick_landmarks=True,
    )
    pvplt.show()


Widget(value='<iframe src="http://localhost:44253/index.html?ui=P_0x7fe28a702550_2&reconnect=auto" class="pyvi…

## Wrap picked points into a `LabeledPoints`


In [11]:
if INTERACTIVE:
    landmarks = get_landmarks()
else:
    assert CACHED_LANDMARKS_TSV, 'set CACHED_LANDMARKS_TSV or INTERACTIVE = True'
    landmarks = cedalion.io.load_tsv(CACHED_LANDMARKS_TSV)

labels = list(landmarks['label'].values)
assert set(labels) == {'Nz', 'Iz', 'Cz', 'Lpa', 'Rpa'}, f'bad labels: {labels}'
display(landmarks)


Magnitude,[[135.82489764260836 134.1195962551057 388.34999788776486] [83.19674841020856 -38.36289544683734 470.02017819060416] [226.49436904915945 -13.33052629089562 371.3030642194328] [144.8723148742921 60.99825004860746 490.2817181736568] [74.20606761144799 16.620827183111732 368.93477871973005]]
Units,millimeter


## Anonymize


In [12]:
surface_anon, landmarks_anon = anonymize_scan(
    surface, landmarks,
    ear_delete_radius_mm=EAR_DELETE_RADIUS_MM,
    landmark_keep_radius_mm=LANDMARK_KEEP_RADIUS_MM,
    eyebrow_offset_mm=EYEBROW_OFFSET_MM,
    cap_z_ceiling_mm=CAP_Z_CEILING_MM,
    head_isolation_radius_mm=HEAD_ISOLATION_RADIUS_MM,
)
stem = Path(SCAN_PATH).stem
print(f'{stem}: {surface.nvertices:,} -> {surface_anon.nvertices:,} verts '
      f'(-{surface.nvertices - surface_anon.nvertices:,})')


Subject13: 943,539 -> 590,551 verts (-352,988)


## Compare original vs anonymized

Both meshes live in the `digitized` frame, so the side-by-side view shares one
coordinate system. The same five landmark spheres are overlaid on each.


In [13]:
stem = Path(SCAN_PATH).stem
lm_colors = {
    'Nz': 'lime', 'Iz': 'magenta', 'Cz': 'cyan',
    'Lpa': 'orange', 'Rpa': 'blue', 'LPA': 'orange', 'RPA': 'blue',
}
n_removed = surface.nvertices - surface_anon.nvertices

orig_vtk = trimesh_to_vtk_polydata(surface.mesh)
anon_vtk = trimesh_to_vtk_polydata(surface_anon.mesh)

lm_pos = landmarks_anon.pint.dequantify().values
lm_lbls = landmarks_anon['label'].values

pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
pvplt.add_mesh(pv.wrap(orig_vtk), rgb=True, smooth_shading=True)
for lbl, pos in zip(lm_lbls, lm_pos):
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos), color=lm_colors.get(lbl, 'yellow'))
pvplt.add_text(f'{stem} Original', position='upper_left', font_size=14)

pvplt.subplot(0, 1)
pvplt.add_mesh(pv.wrap(anon_vtk), rgb=True, smooth_shading=True)
for lbl, pos in zip(lm_lbls, lm_pos):
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos), color=lm_colors.get(lbl, 'yellow'))
pvplt.add_text(
    f'{stem} Anonymized (-{n_removed:,} verts)',
    position='upper_left', font_size=14,
)

pvplt.link_views()
pvplt.show()


Widget(value='<iframe src="http://localhost:44253/index.html?ui=P_0x7fe109fd7890_3&reconnect=auto" class="pyvi…

## Save


In [14]:
written = save_anonymized_scan(surface_anon, OUT_PATH)
for p in written:
    print(f'wrote {p}')


wrote /tmp/Subject13_notebook_test_anon.jpg
wrote /tmp/Subject13_notebook_test_anon.mtl
wrote /tmp/Subject13_notebook_test_anon.obj
